<a id="modelo-knn"></a>

# TC3-54 — K-Nearest Neighbors (KNN)

## Objetivo

Avaliar esta família de modelo de forma independente e reproduzível. O notebook preserva dois momentos do trabalho:

1. **Análise inicial:** configuração básica utilizada na primeira comparação entre famílias.
2. **Refinamento leve:** configurações direcionadas pelos resultados iniciais, sem busca exaustiva.

Todos os cenários usam exatamente a mesma amostra de treino e validação. O conjunto de teste não é acessado.


## Fundamentação e hipótese

Modelo baseado em proximidade. Seu custo de inferência cresce com o volume, portanto a análise usa uma amostra menor e registra essa limitação.

### Critério de comparação

- Métrica principal: ROC-AUC de validação.
- Métricas de apoio: balanced accuracy, F1 e average precision.
- Controle de overfitting: diferença entre ROC-AUC de treino e validação.
- Desempate: menor gap e menor custo computacional.


## Inicialização independente

Esta célula reproduz a configuração compartilhada do notebook principal. Ela prepara ambiente, caminhos e funções comuns sem alterar a lógica canônica desta etapa.


In [1]:
# =============================================================================
# CONFIGURAÇÕES PARA RODAR O NOTEBOOK
# Execute esta célula primeiro. Esta política não altera a metodologia do fluxo.
# =============================================================================
import os
import sys
import gc
import ctypes
import polars as pl
from datetime import datetime, timezone

# Opções do usuário
PERFIL_RECURSOS = "auto"       # auto | baixo | medio | normal | alto
LIMITE_RAM_PERCENTUAL = None    # None = automático; ou inteiro entre 50 e 90
MAX_THREADS = None              # None = automático; ou inteiro >= 1
USAR_GPU = os.getenv("TC3_USAR_GPU", "1").strip().lower() in {"1", "true", "sim", "yes"}                 # requer WSL2 + RAPIDS/cuML + kernel tc3-gpu

REUTILIZAR_ARTEFATOS = False  # False reconstrói; True aceita caches válidos
SALVAR_ARTEFATOS = True       # False usa somente área temporária da sessão


# Protocolo metodologico compartilhado por todas as fases e modelos.
SEED_GLOBAL = 42
EXECUTAR_SHAP = os.getenv("TC3_EXECUTAR_SHAP", "0").strip().lower() in {"1", "true", "sim", "yes"}
AMOSTRA_SHAP = max(20, int(os.getenv("TC3_AMOSTRA_SHAP", "100")))
AMOSTRA_BACKGROUND_SHAP = max(20, int(os.getenv("TC3_BACKGROUND_SHAP", "50")))
CHAVE_GRUPO_GLOBAL = "chave_escola_ano"
N_SPLITS_CV_GLOBAL = 3
N_ITER_BUSCA_GLOBAL = 10
LIMIAR_CLASSIFICACAO_GLOBAL = 0.50
REPETICOES_PERMUTACAO_GLOBAL = 5
REPETICOES_BOOTSTRAP_GLOBAL = 200

N_JOBS_MODELAGEM_GLOBAL = 1
N_SPLITS_HOLDOUT_GLOBAL = 1
SCORING_PRINCIPAL_GLOBAL = "roc_auc"
VERBOSE_BUSCA_GLOBAL = 2
ERRO_BUSCA_GLOBAL = "raise"
RETORNAR_SCORE_TREINO_GLOBAL = True

# Contrato comum de pre-processamento dos modelos.
IMPUTACAO_NUMERICA_GLOBAL = "median"
IMPUTACAO_CATEGORICA_GLOBAL = "most_frequent"
ONE_HOT_DESCONHECIDO_GLOBAL = "ignore"
ONE_HOT_SPARSO_GLOBAL = False  # matriz densa: evita fallback dos modelos para CPU
SPARSE_THRESHOLD_GLOBAL = 0.0
ESCALONADOR_CENTRALIZAR_GLOBAL = False


# Percentuais de amostragem (0 < valor <= 1). Alterar aqui afeta todo o notebook.
# O teste final permanece integral (100%); as amostras menores servem apenas ao
# desenvolvimento, comparacao de modelos e interpretabilidade.
FRACOES_AMOSTRAGEM = {
    "eda_correlacao": 0.05,
    "validacao_preparacao": 0.02,
    "fase5_modelos": 0.10,
    "fase5_knn": 0.02,  # KNN tem custo de memoria e predicao maior.
    "fase6_desenvolvimento": 0.12,
    "fase6_validacao_interna": 0.25,
    "fase7_desenvolvimento": 1.00,
    "fase7_teste": 1.00,
    "interpretacao": 0.035,
    "bootstrap": 0.17,
}

for _nome, _fracao in FRACOES_AMOSTRAGEM.items():
    if not 0 < _fracao <= 1:
        raise ValueError(f"Fracao invalida em FRACOES_AMOSTRAGEM[{_nome!r}]: {_fracao}")


def tamanho_por_fracao(total, fracao):
    """Converte uma proporcao em quantidade reproduzivel, sem exceder o total."""
    if total < 1:
        raise ValueError("Nao e possivel amostrar uma particao vazia.")
    return min(total, max(1, round(total * fracao)))


def materializar_split_percentual(dados, contrato, nome_split, fracao, seed, materializador):
    """Materializa uma fracao de um split e informa claramente o universo usado."""
    import polars as pl

    coluna_split = contrato["controle_split"]
    total = (
        dados.filter(pl.col(coluna_split) == nome_split)
        .select(pl.len().alias("total"))
        .collect()
        .item()
    )
    quantidade = tamanho_por_fracao(total, fracao)
    X, y = materializador(dados, contrato, nome_split, limit=quantidade, seed=seed)
    print(f"{nome_split}: {quantidade:,}/{total:,} registros ({fracao:.1%})")
    return X, y


def _estado_memoria_sistema():
    """Retorna RAM total e disponível em bytes."""
    valores = {}
    try:
        with open("/proc/meminfo", encoding="utf-8") as arquivo:
            for linha in arquivo:
                chave, valor = linha.split(":", 1)
                valores[chave] = int(valor.strip().split()[0]) * 1024
        return valores["MemTotal"], valores.get("MemAvailable", valores.get("MemFree", 0))
    except (OSError, KeyError, ValueError):
        try:
            return (
                os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
                os.sysconf("SC_AVPHYS_PAGES") * os.sysconf("SC_PAGE_SIZE"),
            )
        except (AttributeError, OSError, ValueError):
            return None, None


RAM_TOTAL_BYTES, RAM_DISPONIVEL_BYTES = _estado_memoria_sistema()
RAM_TOTAL_GB = RAM_TOTAL_BYTES / 1024**3 if RAM_TOTAL_BYTES else 0.0
CPUS_LOGICAS = os.cpu_count() or 1

if RAM_TOTAL_GB and RAM_TOTAL_GB <= 4.5:
    _perfil_detectado = "baixo"
elif RAM_TOTAL_GB and RAM_TOTAL_GB <= 8.5:
    _perfil_detectado = "medio"
elif RAM_TOTAL_GB and RAM_TOTAL_GB <= 16.5:
    _perfil_detectado = "normal"
else:
    _perfil_detectado = "alto"

_perfis = {
    "baixo": (60, 1),
    "medio": (70, min(2, CPUS_LOGICAS)),
    "normal": (75, min(4, CPUS_LOGICAS)),
    "alto": (80, min(8, max(1, CPUS_LOGICAS // 2))),
}
PERFIL_ATIVO = _perfil_detectado if PERFIL_RECURSOS == "auto" else PERFIL_RECURSOS.lower()
if PERFIL_ATIVO not in _perfis:
    raise ValueError("PERFIL_RECURSOS deve ser auto, baixo, medio, normal ou alto.")

_limite_perfil, _threads_perfil = _perfis[PERFIL_ATIVO]
LIMITE_RAM_ATIVO = _limite_perfil if LIMITE_RAM_PERCENTUAL is None else int(LIMITE_RAM_PERCENTUAL)
THREADS_MAXIMAS = _threads_perfil if MAX_THREADS is None else int(MAX_THREADS)
if not 50 <= LIMITE_RAM_ATIVO <= 90:
    raise ValueError("LIMITE_RAM_PERCENTUAL deve ficar entre 50 e 90.")
if THREADS_MAXIMAS < 1:
    raise ValueError("MAX_THREADS deve ser pelo menos 1.")
THREADS_MAXIMAS = min(THREADS_MAXIMAS, CPUS_LOGICAS)

# Definido antes de NumPy, Polars, scikit-learn e cuML serem importados.
for _variavel, _valor in {
    "OMP_NUM_THREADS": str(THREADS_MAXIMAS),
    "OPENBLAS_NUM_THREADS": str(THREADS_MAXIMAS),
    "MKL_NUM_THREADS": str(THREADS_MAXIMAS),
    "NUMEXPR_NUM_THREADS": str(THREADS_MAXIMAS),
    "VECLIB_MAXIMUM_THREADS": str(THREADS_MAXIMAS),
    "POLARS_MAX_THREADS": str(THREADS_MAXIMAS),
    "MALLOC_ARENA_MAX": "2",
}.items():
    os.environ[_variavel] = _valor

HISTORICO_RECURSOS = []
_estado_recursos = {"nivel": None, "threads": None, "contexto_threads": None}


def _aplicar_threads(limite):
    """Limita pools BLAS/OpenMP já carregados; Polars usa o limite inicial."""
    limite = max(1, min(int(limite), THREADS_MAXIMAS))
    if _estado_recursos["threads"] == limite:
        return
    anterior = _estado_recursos.get("contexto_threads")
    if anterior is not None:
        try:
            anterior.__exit__(None, None, None)
        except Exception:
            pass
    try:
        from threadpoolctl import threadpool_limits
        contexto = threadpool_limits(limits=limite)
        contexto.__enter__()
    except Exception:
        contexto = None
    _estado_recursos["contexto_threads"] = contexto
    _estado_recursos["threads"] = limite


def _liberar_memoria_global(agressiva=False):
    """Libera caches transitórios sem apagar variáveis usadas pelo fluxo."""
    shell = get_ipython()
    if shell is not None:
        for nome in ("_", "__", "___"):
            shell.user_ns[nome] = None
    gc.collect()
    modulo_cupy = sys.modules.get("cupy")
    if modulo_cupy is not None:
        try:
            modulo_cupy.get_default_memory_pool().free_all_blocks()
            modulo_cupy.get_default_pinned_memory_pool().free_all_blocks()
        except Exception:
            pass
    if agressiva:
        modulo_plt = sys.modules.get("matplotlib.pyplot")
        if modulo_plt is not None:
            try:
                modulo_plt.close("all")
            except Exception:
                pass
        try:
            ctypes.CDLL("libc.so.6").malloc_trim(0)
        except (OSError, AttributeError):
            pass


def _gerenciar_recursos(*_args, **_kwargs):
    """Reduz paralelismo progressivamente e nunca interrompe uma célula."""
    total, disponivel = _estado_memoria_sistema()
    if not total:
        return
    uso = 100.0 * (total - disponivel) / total
    margem = LIMITE_RAM_ATIVO - uso
    if margem > 15:
        nivel, threads, limpar = "normal", THREADS_MAXIMAS, False
    elif margem > 7:
        nivel, threads, limpar = "moderado", max(1, THREADS_MAXIMAS // 2), True
    else:
        nivel, threads, limpar = "conservador", 1, True

    _aplicar_threads(threads)
    if limpar:
        _liberar_memoria_global(agressiva=margem <= 0)

    if nivel != _estado_recursos["nivel"]:
        registro = {
            "instante_utc": datetime.now(timezone.utc).isoformat(),
            "nivel": nivel,
            "threads": threads,
            "ram_usada_percentual": round(uso, 1),
            "ram_disponivel_gb": round(disponivel / 1024**3, 2),
        }
        HISTORICO_RECURSOS.append(registro)
        print(
            f"[recursos] nível={nivel}; threads={threads}; "
            f"RAM usada={uso:.1f}%; livre={disponivel / 1024**3:.1f} GB."
        )
        _estado_recursos["nivel"] = nivel


ipython = get_ipython()
if ipython is not None:
    # Impede callbacks duplicados ao reexecutar a configuração.
    for _atributo in ("_tc3_memory_guard", "_tc3_resource_guard"):
        _guard_anterior = getattr(ipython, _atributo, None)
        if _guard_anterior is not None:
            for _evento, _funcao in _guard_anterior:
                try:
                    ipython.events.unregister(_evento, _funcao)
                except (ValueError, KeyError):
                    pass
    _callbacks_recursos = [
        ("pre_run_cell", _gerenciar_recursos),
        ("post_run_cell", _gerenciar_recursos),
    ]
    for _evento, _funcao in _callbacks_recursos:
        ipython.events.register(_evento, _funcao)
    ipython._tc3_resource_guard = _callbacks_recursos

_gerenciar_recursos()
print(
    "Configuração automática ativa: "
    f"perfil={PERFIL_ATIVO}; RAM={RAM_TOTAL_GB:.1f} GB; CPUs={CPUS_LOGICAS}; "
    f"limite preventivo={LIMITE_RAM_ATIVO}%; threads máximas={THREADS_MAXIMAS}."
)

# Backend GPU: ativado antes de qualquer importação do scikit-learn.
cuda_conda = os.path.join(sys.prefix, "targets", "x86_64-linux")
if os.path.exists(os.path.join(cuda_conda, "include", "cuda.h")):
    os.environ.setdefault("CUDA_PATH", cuda_conda)
if USAR_GPU:
    try:
        from cuml.accel import install
        install(log_level="info")
        import cupy as cp
        dispositivo = cp.cuda.runtime.getDeviceProperties(0)
        nome_gpu = dispositivo["name"].decode() if isinstance(dispositivo["name"], bytes) else str(dispositivo["name"])
        memoria_gpu_gb = cp.cuda.runtime.memGetInfo()[1] / 1024**3
        BACKEND_ML = f"RAPIDS cuML — {nome_gpu} ({memoria_gpu_gb:.1f} GB de VRAM)"
    except Exception as exc:
        raise RuntimeError("GPU/RAPIDS indisponível. Abra no WSL2 e selecione o kernel tc3-gpu.") from exc
else:
    BACKEND_ML = "scikit-learn CPU"
print("Backend de Machine Learning:", BACKEND_ML)

from pathlib import Path
import hashlib
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

def localizar_raiz(inicio: Path) -> Path:
    for candidato in [inicio, *inicio.parents]:
        if (candidato / "dados").is_dir() and (candidato / "notebooks").is_dir():
            return candidato
    raise FileNotFoundError("Execute este notebook dentro do repositório tech_challenge_3.")

PROJECT_ROOT = localizar_raiz(Path.cwd().resolve())
print("Raiz do projeto:", PROJECT_ROOT)


# Contrato único de diretórios. Fontes sempre são lidas da área imutável.
import tempfile

FONTES_ORIGINAIS_DIR = PROJECT_ROOT / "dados" / "fontes_originais"
_TEMP_BASE = PROJECT_ROOT / ".execucao_temporaria"
_TEMP_BASE.mkdir(parents=True, exist_ok=True)
if SALVAR_ARTEFATOS:
    RAIZ_ARTEFATOS = PROJECT_ROOT
    _DIRETORIO_TEMPORARIO = None
else:
    _DIRETORIO_TEMPORARIO = tempfile.TemporaryDirectory(prefix="tc3_", dir=_TEMP_BASE)
    RAIZ_ARTEFATOS = Path(_DIRETORIO_TEMPORARIO.name)

DADOS_PROCESSADOS_DIR = RAIZ_ARTEFATOS / "dados" / "dados_processados"
DADOS_ANALITICOS_DIR = RAIZ_ARTEFATOS / "dados" / "dados_analiticos"
RESULTADOS_DIR = RAIZ_ARTEFATOS / "resultados"
RESULTADOS_FIGURAS_DIR = RESULTADOS_DIR / "figuras"
RESULTADOS_TABELAS_DIR = RESULTADOS_DIR / "tabelas"
RESULTADOS_METRICAS_DIR = RESULTADOS_DIR / "metricas"
RESULTADOS_RELATORIOS_DIR = RESULTADOS_DIR / "relatorios"
MODELOS_DIR = RAIZ_ARTEFATOS / "modelos"

for _diretorio_saida in (
    DADOS_PROCESSADOS_DIR, DADOS_ANALITICOS_DIR, RESULTADOS_FIGURAS_DIR,
    RESULTADOS_TABELAS_DIR, RESULTADOS_METRICAS_DIR,
    RESULTADOS_RELATORIOS_DIR, MODELOS_DIR,
):
    _diretorio_saida.mkdir(parents=True, exist_ok=True)

def deve_recalcular(caminho):
    """True quando o cache deve ser ignorado ou ainda não existe."""
    return not REUTILIZAR_ARTEFATOS or not Path(caminho).exists()


def publicar_parquet_particionado(frame, destino, *, particionar_por):
    """Publica partições sequencialmente e troca o diretório de forma atômica.

    O ``write_parquet(partition_by=...)`` do Polars prepara várias partições em
    paralelo e pode duplicar o pico de RAM de bases já materializadas. Esta
    implementação mantém apenas uma partição adicional em memória por vez.
    """
    import gc
    import shutil

    destino = Path(destino)
    temporario = destino.with_name(destino.name + "__publicacao_tmp")
    anterior = destino.with_name(destino.name + "__publicacao_anterior")
    for caminho in (temporario, anterior):
        if caminho.exists():
            shutil.rmtree(caminho)
    temporario.mkdir(parents=True, exist_ok=True)

    lazy = frame if isinstance(frame, pl.LazyFrame) else frame.lazy()
    schema = lazy.collect_schema()
    if particionar_por not in schema.names():
        raise ValueError(f"Coluna de particionamento ausente: {particionar_por}")

    valores = (
        lazy.select(pl.col(particionar_por).unique())
        .collect(engine="streaming")
        .get_column(particionar_por)
        .to_list()
    )
    valores = sorted(valores, key=lambda valor: (valor is None, str(valor)))
    if not valores:
        raise RuntimeError("Não há valores para particionar a publicação.")

    arquivos_novos = []
    for indice, valor in enumerate(valores):
        valor_hive = "__HIVE_DEFAULT_PARTITION__" if valor is None else str(valor)
        if any(separador in valor_hive for separador in ("/", "\\")):
            raise ValueError(f"Valor de partição inválido: {valor_hive!r}")
        diretorio_particao = temporario / f"{particionar_por}={valor_hive}"
        diretorio_particao.mkdir(parents=True, exist_ok=True)
        arquivo = diretorio_particao / f"part-{indice:05d}.parquet"
        predicado = pl.col(particionar_por).is_null() if valor is None else pl.col(particionar_por) == pl.lit(valor)
        lazy.filter(predicado).sink_parquet(
            arquivo,
            compression="zstd",
            statistics=True,
            row_group_size=100_000,
            mkdir=True,
            engine="streaming",
        )
        if not arquivo.is_file() or arquivo.stat().st_size == 0:
            raise RuntimeError(f"Partição não foi gravada corretamente: {arquivo}")
        arquivos_novos.append(arquivo)
        gc.collect()

    if destino.exists():
        destino.replace(anterior)
    try:
        try:
            temporario.replace(destino)
        except OSError:
            # Volumes Docker montados do Windows podem bloquear rename de diretórios.
            # O fallback copia a publicação pronta e remove a área temporária.
            shutil.copytree(temporario, destino)
            shutil.rmtree(temporario)
    except Exception:
        if destino.exists():
            shutil.rmtree(destino)
        if anterior.exists():
            anterior.replace(destino)
        raise
    if anterior.exists():
        shutil.rmtree(anterior)
    return len(arquivos_novos)


def salvar_figura_eda(figura, nome):
    """Exporta uma figura exploratória sem alterar sua exibição no notebook."""
    destino = RESULTADOS_FIGURAS_DIR / "fluxo_exploratorio" / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    figura.savefig(destino, dpi=180, bbox_inches="tight")
    return destino

print("Fontes originais:", FONTES_ORIGINAIS_DIR)
print("Destino dos artefatos:", RAIZ_ARTEFATOS)
print("Reutilizar artefatos:", REUTILIZAR_ARTEFATOS, "| Salvar:", SALVAR_ARTEFATOS)

# Imports compartilhados que, no fluxo completo, são carregados por etapas anteriores.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


[recursos] nível=normal; threads=4; RAM usada=38.9%; livre=9.4 GB.
Configuração automática ativa: perfil=normal; RAM=15.5 GB; CPUs=20; limite preventivo=75%; threads máximas=4.
[cuml.accel] Accelerator installed.
Backend de Machine Learning: RAPIDS cuML — NVIDIA GeForce RTX 3050 6GB Laptop GPU (6.0 GB de VRAM)
Raiz do projeto: /workspace
Fontes originais: /workspace/dados/fontes_originais
Destino dos artefatos: /workspace
Reutilizar artefatos: False | Salvar: True


### Contrato mínimo de modelagem

As funções abaixo são as mesmas do fluxo principal e permitem executar este modelo isoladamente.


In [2]:
# Funções compartilhadas copiadas sem alteração das etapas TC3-49 e TC3-52.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pathlib import Path
import json
import time

import numpy as np
import polars as pl
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)


def load_modeling_data(_project_root=None):
    """Carrega contrato e LazyFrame sem depender de scripts externos."""
    manifest_path = DADOS_PROCESSADOS_DIR / "preparado" / "manifesto_base_modelagem_com_chave_escola.json"
    if not manifest_path.exists():
        raise FileNotFoundError("Execute primeiro a preparação da base e a publicação do manifesto.")
    contract = json.loads(manifest_path.read_text(encoding="utf-8"))
    data_glob = PROJECT_ROOT / Path(contract["padrao_leitura"])
    data = pl.scan_parquet(str(data_glob), hive_partitioning=True)
    schema = data.collect_schema()
    required = contract["features_modelagem"] + [contract["target"], contract["controle_split"]]
    missing = sorted(set(required) - set(schema.names()))
    if missing:
        raise KeyError(f"Colunas do contrato ausentes na base: {missing}")
    return data, contract


def materialize_split(data, contract, split, *, limit, seed):
    """Materializa somente treino ou validação, com amostragem reprodutível."""
    features = contract["features_modelagem"]
    target = contract["target"]
    split_column = contract["controle_split"]
    frame = data.filter(pl.col(split_column) == split)
    if limit is not None:
        frame = (
            frame.with_columns(pl.col("id_aluno").hash(seed=seed).alias("_sample_key"))
            .sort("_sample_key").head(limit).drop("_sample_key")
        )
    frame = frame.select(features + [target]).collect()
    if frame.is_empty():
        raise ValueError(f"O split {split!r} ficou vazio.")
    return frame.select(features), frame[target].to_numpy()


def _prediction_score(model, X):
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X))[:, 1]
    if hasattr(model, "decision_function"):
        return np.asarray(model.decision_function(X))
    return np.asarray(model.predict(X), dtype=float)


def evaluate_classifier(model, X_train, y_train, X_valid, y_valid, *, model_id, model_name, execution_mode, config):
    started = time.perf_counter()
    model.fit(X_train, y_train)
    train_seconds = time.perf_counter() - started
    started = time.perf_counter()
    prediction = np.asarray(model.predict(X_valid))
    score = _prediction_score(model, X_valid)
    inference_seconds = time.perf_counter() - started
    metrics = {
        "accuracy": float(accuracy_score(y_valid, prediction)),
        "balanced_accuracy": float(balanced_accuracy_score(y_valid, prediction)),
        "precision": float(precision_score(y_valid, prediction, zero_division=0)),
        "recall": float(recall_score(y_valid, prediction, zero_division=0)),
        "f1": float(f1_score(y_valid, prediction, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_valid, score)),
        "average_precision": float(average_precision_score(y_valid, score)),
    }
    return {
        "schema_resultado": 1,
        "modelo": {"id": model_id, "nome": model_name},
        "execucao": {"modo": execution_mode, "split_avaliacao": "validacao"},
        "dados": {"registros_treino": int(len(y_train)), "registros_validacao": int(len(y_valid))},
        "configuracao": config,
        "metricas": metrics,
        "matriz_confusao": confusion_matrix(y_valid, prediction).tolist(),
        "tempos_segundos": {"treino": float(train_seconds), "inferencia": float(inference_seconds)},
        "limitacoes": [
            "São Paulo possui registros somente em 2024.",
            "O teste permanece reservado até a avaliação final.",
        ],
    }


def export_result(result, _project_root=None):
    result_dir = RESULTADOS_METRICAS_DIR / "fase_5"
    result_dir.mkdir(parents=True, exist_ok=True)
    output = result_dir / f"{result['modelo']['id']}_resultado.json"
    temporario = output.with_suffix(output.suffix + ".tmp")
    temporario.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    temporario.replace(output)
    return output


def build_pipeline(estimator, contract, *, kind):
    from sklearn.preprocessing import OrdinalEncoder
    features = contract["features_modelagem"]
    categoricas = ["sigla_uf", "rede"]
    numericas = [c for c in features if c not in categoricas]
    indices_categoricas = [features.index(c) for c in categoricas]
    indices_numericas = [features.index(c) for c in numericas]
    if kind in {"linear", "tree"}:
        categorical = Pipeline([
            ("imputer", SimpleImputer(strategy=IMPUTACAO_CATEGORICA_GLOBAL)),
            ("encoder", OneHotEncoder(handle_unknown=ONE_HOT_DESCONHECIDO_GLOBAL, sparse_output=ONE_HOT_SPARSO_GLOBAL, dtype=np.float32)),
        ])
        numeric_steps = [("imputer", SimpleImputer(strategy=IMPUTACAO_NUMERICA_GLOBAL))]
        if kind == "linear":
            numeric_steps.append(("scaler", StandardScaler(with_mean=ESCALONADOR_CENTRALIZAR_GLOBAL)))
        numeric = Pipeline(numeric_steps)
    elif kind == "hist":
        categorical = Pipeline([
            ("imputer", SimpleImputer(strategy=IMPUTACAO_CATEGORICA_GLOBAL)),
            ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
        ])
        numeric = Pipeline([("imputer", SimpleImputer(strategy=IMPUTACAO_NUMERICA_GLOBAL))])
    else:
        raise ValueError(f"Tipo de pipeline desconhecido: {kind}")
    model_preprocessor = ColumnTransformer(
        [("categorical", categorical, indices_categoricas), ("numeric", numeric, indices_numericas)],
        sparse_threshold=SPARSE_THRESHOLD_GLOBAL,
    )
    return Pipeline([("preprocessor", model_preprocessor), ("model", estimator)])

MODELOS_GPU_OBRIGATORIOS = {"LogisticRegression", "KNeighborsClassifier", "LinearSVC", "RandomForestClassifier"}
def validar_backend_modelo(modelo):
    estimador = modelo.named_steps.get("model", modelo) if hasattr(modelo, "named_steps") else modelo
    nome = type(estimador).__name__
    executou_gpu = getattr(estimador, "_gpu", None) is not None
    if USAR_GPU:
        if nome in MODELOS_GPU_OBRIGATORIOS and not executou_gpu:
            raise RuntimeError(f"{nome} não executou na GPU. Interrompa: não publique este resultado como GPU.")
        backend = "GPU" if executou_gpu else "CPU esperada"
    else:
        backend = "CPU"
    print(f"Backend validado — {nome}: {backend}")


In [3]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import polars as pl
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)

# Funções de dados definidas no próprio notebook (TC3-52).
dados, contrato = load_modeling_data()
FEATURES = contrato["features_modelagem"]
TARGET = contrato["target"]
print(contrato["produto"], contrato["registros"], "registros", len(FEATURES), "features")
print("Splits permitidos nesta etapa: treino e validacao. O teste permanece reservado.")

from sklearn.neighbors import KNeighborsClassifier


base_modelagem_final_tc3_com_chave_escola 3867974 registros 18 features
Splits permitidos nesta etapa: treino e validacao. O teste permanece reservado.


## K-Nearest Neighbors (KNN) — amostra controlada

Esta etapa pertence à **Fase 5 — comparação inicial** e avalia o modelo **K-Nearest Neighbors (KNN)**. Treino e validação são amostrados separadamente, preservando os splits já definidos e mantendo o teste totalmente reservado. A proporção utilizada vem de `FRACOES_AMOSTRAGEM["fase5_knn"]`, declarada na configuração inicial.


In [4]:
SEED = SEED_GLOBAL
FRACAO_MODELO = FRACOES_AMOSTRAGEM["fase5_knn"]

X_train, y_train = materializar_split_percentual(
    dados, contrato, "treino", FRACAO_MODELO, SEED, materialize_split
)
X_valid, y_valid = materializar_split_percentual(
    dados, contrato, "validacao", FRACAO_MODELO, SEED + 1, materialize_split
)

print("Modelo em avaliacao: K-Nearest Neighbors (KNN)")
print("Taxa positiva treino:", round(float(np.mean(y_train)), 4))
print("Taxa positiva validacao:", round(float(np.mean(y_valid)), 4))


treino: 54,105/2,705,236 registros (2.0%)
validacao: 11,623/581,147 registros (2.0%)
Modelo em avaliacao: K-Nearest Neighbors (KNN)
Taxa positiva treino: 0.5125
Taxa positiva validacao: 0.5123


In [5]:
def pontuar(modelo, X, y):
    predicao = np.asarray(modelo.predict(X))
    if hasattr(modelo, "predict_proba"):
        escore = np.asarray(modelo.predict_proba(X))[:, 1]
    else:
        escore = np.asarray(modelo.decision_function(X))
    return {
        "accuracy": float(accuracy_score(y, predicao)),
        "balanced_accuracy": float(balanced_accuracy_score(y, predicao)),
        "precision": float(precision_score(y, predicao, zero_division=0)),
        "recall": float(recall_score(y, predicao, zero_division=0)),
        "f1": float(f1_score(y, predicao, zero_division=0)),
        "roc_auc": float(roc_auc_score(y, escore)),
        "average_precision": float(average_precision_score(y, escore)),
        "matriz_confusao": confusion_matrix(y, predicao).tolist(),
    }

def avaliar_cenario(nome, estimador, tipo_pipeline, parametros):
    modelo = build_pipeline(estimador, contrato, kind=tipo_pipeline)
    inicio = time.perf_counter()
    modelo.fit(X_train, y_train)
    validar_backend_modelo(modelo)
    tempo_treino = time.perf_counter() - inicio
    metricas_treino = pontuar(modelo, X_train, y_train)
    inicio = time.perf_counter()
    metricas_validacao = pontuar(modelo, X_valid, y_valid)
    tempo_inferencia = time.perf_counter() - inicio
    return {
        "cenario": nome,
        "parametros": parametros,
        "modelo_treinado": modelo,
        "metricas_treino": metricas_treino,
        "metricas_validacao": metricas_validacao,
        "gap_roc_auc": metricas_treino["roc_auc"] - metricas_validacao["roc_auc"],
        "tempo_treino_s": tempo_treino,
        "tempo_inferencia_s": tempo_inferencia,
    }


## K-Nearest Neighbors (KNN) — 1. Análise inicial — configuração básica

A configuração inicial usa 25 vizinhos e pesos uniformes. Essa escolha é compatível tanto com scikit-learn em CPU quanto com a aceleração opcional via RAPIDS cuML.


In [6]:
resultados = []
PARAMS_INICIAL = {'n_neighbors': 25, 'weights': 'uniform', 'n_jobs': N_JOBS_MODELAGEM_GLOBAL}
resultado_inicial = avaliar_cenario(
    "inicial",
    KNeighborsClassifier(**PARAMS_INICIAL),
    'linear',
    PARAMS_INICIAL,
)
resultados.append(resultado_inicial)
display(pl.DataFrame([{
    "cenario": resultado_inicial["cenario"],
    "roc_auc_validacao": resultado_inicial["metricas_validacao"]["roc_auc"],
    "balanced_accuracy_validacao": resultado_inicial["metricas_validacao"]["balanced_accuracy"],
    "f1_validacao": resultado_inicial["metricas_validacao"]["f1"],
    "gap_roc_auc": resultado_inicial["gap_roc_auc"],
    "tempo_treino_s": resultado_inicial["tempo_treino_s"],
}]))
print("Matriz de confusão:", resultado_inicial["metricas_validacao"]["matriz_confusao"])


[cuml.accel] `KNeighborsClassifier.fit` ran on GPU
Backend validado — KNeighborsClassifier: GPU
[cuml.accel] `KNeighborsClassifier.predict` ran on GPU
[cuml.accel] `KNeighborsClassifier.predict_proba` ran on GPU
[cuml.accel] `KNeighborsClassifier.predict` ran on GPU
[cuml.accel] `KNeighborsClassifier.predict_proba` ran on GPU


cenario,roc_auc_validacao,balanced_accuracy_validacao,f1_validacao,gap_roc_auc,tempo_treino_s
str,f64,f64,f64,f64,f64
"""inicial""",0.615332,0.587055,0.595379,0.060888,0.166765


Matriz de confusão: [[3295, 2374], [2424, 3530]]


## K-Nearest Neighbors (KNN) — 2. Refinamento leve — análise direcionada

Comparamos menor e maior suavização, usando 15 e 35 vizinhos, para observar viés, variância e custo.

O objetivo não é encontrar artificialmente o melhor resultado, mas verificar se pequenos ajustes coerentes melhoram generalização ou custo.


In [7]:
PARAMS_1 = {'n_neighbors': 15, 'weights': 'uniform', 'n_jobs': N_JOBS_MODELAGEM_GLOBAL}
resultados.append(avaliar_cenario(
    'refinado_k15',
    KNeighborsClassifier(**PARAMS_1),
    'linear',
    PARAMS_1,
))
PARAMS_2 = {'n_neighbors': 35, 'weights': 'uniform', 'n_jobs': N_JOBS_MODELAGEM_GLOBAL}
resultados.append(avaliar_cenario(
    'refinado_k35',
    KNeighborsClassifier(**PARAMS_2),
    'linear',
    PARAMS_2,
))


[cuml.accel] `KNeighborsClassifier.fit` ran on GPU
Backend validado — KNeighborsClassifier: GPU
[cuml.accel] `KNeighborsClassifier.predict` ran on GPU
[cuml.accel] `KNeighborsClassifier.predict_proba` ran on GPU
[cuml.accel] `KNeighborsClassifier.predict` ran on GPU
[cuml.accel] `KNeighborsClassifier.predict_proba` ran on GPU
[cuml.accel] `KNeighborsClassifier.fit` ran on GPU
Backend validado — KNeighborsClassifier: GPU
[cuml.accel] `KNeighborsClassifier.predict` ran on GPU
[cuml.accel] `KNeighborsClassifier.predict_proba` ran on GPU
[cuml.accel] `KNeighborsClassifier.predict` ran on GPU
[cuml.accel] `KNeighborsClassifier.predict_proba` ran on GPU


## K-Nearest Neighbors (KNN) — 3. Comparação dos cenários


In [8]:
comparacao = pl.DataFrame([
    {
        "cenario": r["cenario"],
        "roc_auc_treino": r["metricas_treino"]["roc_auc"],
        "roc_auc_validacao": r["metricas_validacao"]["roc_auc"],
        "gap_roc_auc": r["gap_roc_auc"],
        "balanced_accuracy_validacao": r["metricas_validacao"]["balanced_accuracy"],
        "f1_validacao": r["metricas_validacao"]["f1"],
        "average_precision_validacao": r["metricas_validacao"]["average_precision"],
        "tempo_treino_s": r["tempo_treino_s"],
        "tempo_inferencia_s": r["tempo_inferencia_s"],
    }
    for r in resultados
]).sort(["roc_auc_validacao", "balanced_accuracy_validacao", "f1_validacao"], descending=True)
display(comparacao)

melhor_nome = comparacao.row(0, named=True)["cenario"]
melhor = next(r for r in resultados if r["cenario"] == melhor_nome)
print("Cenário selecionado:", melhor_nome)
print("Parâmetros selecionados:", melhor["parametros"])


cenario,roc_auc_treino,roc_auc_validacao,gap_roc_auc,balanced_accuracy_validacao,f1_validacao,average_precision_validacao,tempo_treino_s,tempo_inferencia_s
str,f64,f64,f64,f64,f64,f64,f64,f64
"""refinado_k35""",0.668047,0.617201,0.050847,0.587358,0.607833,0.615048,0.098736,1.896315
"""inicial""",0.67622,0.615332,0.060888,0.587055,0.595379,0.608926,0.166765,1.854367
"""refinado_k15""",0.682909,0.610505,0.072404,0.582549,0.589975,0.602129,0.115816,1.860367


Cenário selecionado: refinado_k35
Parâmetros selecionados: {'n_neighbors': 35, 'weights': 'uniform', 'n_jobs': 1}


## K-Nearest Neighbors (KNN) — 4. Exportação e rastreabilidade

O JSON detalhado preserva todos os cenários. O JSON padrão do modelo contém o cenário selecionado e continua compatível com o notebook TC3-59.


In [9]:
RESULT_DIR = RESULTADOS_DIR / "metricas" / "fase_5"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def serializar_cenario(r):
    return {k: v for k, v in r.items() if k != "modelo_treinado"}

detalhado = {
    "schema_resultado": 2,
    "modelo": {"id": "TC3-54", "nome": 'K-Nearest Neighbors (KNN)'},
    "metodologia": {
        "comparacao_controlada": True,
        "split_avaliacao": "validacao",
        "teste_utilizado": False,
        "registros_treino": len(y_train),
        "registros_validacao": len(y_valid),
        "seed": SEED,
    },
    "cenario_selecionado": melhor["cenario"],
    "cenarios": [serializar_cenario(r) for r in resultados],
}

resultado_padrao = {
    "schema_resultado": 2,
    "modelo": detalhado["modelo"],
    "execucao": {"modo": "comparacao_inicial_refinada", "split_avaliacao": "validacao"},
    "dados": {"registros_treino": len(y_train), "registros_validacao": len(y_valid)},
    "configuracao": {
        "seed": SEED,
        "cenario_selecionado": melhor["cenario"],
        "parametros": melhor["parametros"],
    },
    "metricas": {k: v for k, v in melhor["metricas_validacao"].items() if k != "matriz_confusao"},
    "matriz_confusao": melhor["metricas_validacao"]["matriz_confusao"],
    "tempos_segundos": {
        "treino": melhor["tempo_treino_s"],
        "inferencia": melhor["tempo_inferencia_s"],
    },
    "limitacoes": [
        "São Paulo possui registros somente em 2024; não há validação temporal de SP para 2023.",
        "O split de teste permanece reservado para avaliação final após seleção e tuning.",
    ],
}

detalhado_path = RESULT_DIR / "TC3-54_comparacao_detalhada.json"
padrao_path = RESULT_DIR / "TC3-54_resultado.json"
detalhado_path.write_text(json.dumps(detalhado, ensure_ascii=False, indent=2), encoding="utf-8")
padrao_path.write_text(json.dumps(resultado_padrao, ensure_ascii=False, indent=2), encoding="utf-8")
print("Comparação detalhada:", detalhado_path)
print("Resultado para consolidação:", padrao_path)


Comparação detalhada: /workspace/resultados/metricas/fase_5/TC3-54_comparacao_detalhada.json
Resultado para consolidação: /workspace/resultados/metricas/fase_5/TC3-54_resultado.json


## K-Nearest Neighbors (KNN) — 5. Conclusão do modelo

Ao concluir a execução, registre na apresentação:

- comportamento da configuração inicial;
- motivo dos ajustes testados;
- ganho ou perda do refinamento;
- evidência de overfitting pelo gap;
- custo de treino e inferência;
- justificativa para manter ou descartar esta família.

O resultado ainda é de validação. Ele não constitui a avaliação final no conjunto de teste.
